In [1]:
from pathlib import Path
import pandas as pd
import re
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Carico i CSV

In [2]:
df_duke = pd.read_csv(FILE_PATH / "duke_lesions_radiomic_medsam.csv")
df_ambl = pd.read_csv(FILE_PATH / "ambl_lesions_radiomic_medsam.csv")

print("DUKE shape:", df_duke.shape)
print("AMBL shape:", df_ambl.shape)


DUKE shape: (291, 109)
AMBL shape: (82, 111)


# Definisco il target di interesse

In [3]:
target_row = {
    "PR" : ["PR", "PR [SII]"]
}

# Cerco a colonna PR nei csv

In [4]:
def get_target_column (df, target_aliases):
    for col in target_aliases:
        if col in df.columns:
            return col
    raise ValueError(f"Nessuna colonna target trovata tra: {target_aliases}")

# Binarizzo ambl

In [5]:
df_ambl["PR_binario"] = (
        pd.to_numeric(df_ambl["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

# Applico la funzione

In [6]:
target_duke = get_target_column(df_duke, target_row["PR"])
target_ambl = "PR_binario"

print("Target PR DUKE:", target_duke)
print("Target PR AMBL:", target_ambl)

Target PR DUKE: PR
Target PR AMBL: PR_binario


# Controllo delle classi

In [7]:
print("\nDistribuzione PR DUKE:")
print(df_duke[target_duke].value_counts(dropna=False))

print("\nDistribuzione PR AMBL:")
print(df_ambl[target_ambl].value_counts(dropna=False))



Distribuzione PR DUKE:
PR
0    157
1    134
Name: count, dtype: int64

Distribuzione PR AMBL:
PR_binario
0    46
1    36
Name: count, dtype: int64


# Preparo le feature

In [8]:
def prepare_features(df, target_col):
    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign",
        "GRADE", "isTN", "Breast", target_col
    ]

    X = df.drop(columns=features_to_drop, errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.mean(numeric_only=True))
    X.columns = [re.sub(r"\[|\]|<", "", c) for c in X.columns]

    return X

# Definisco i ruoli

In [9]:
df_train_source = df_ambl
df_test_external = df_duke

target_train = target_ambl     # PR_binario
target_test  = target_duke     # PR


# Split


In [10]:
X_full = prepare_features(df_train_source, target_train)
y_full = df_train_source[target_train]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X_full, y_full))

train_set = df_train_source.iloc[train_idx].reset_index(drop=True)
val_internal = df_train_source.iloc[val_idx].reset_index(drop=True)

print("Train:", train_set.shape)
print("Val:", val_internal.shape)


Train: (65, 112)
Val: (17, 112)


# BOOTSTRAP solo del training

creo un nuovo training set con la stessa dimensione del training originale ma con campioni ripetuti e altri esclusi

In [11]:
train_bootstrap = (
    train_set
    .sample(n=len(train_set), replace=True, random_state=42)
    .reset_index(drop=True)
)

print("Bootstrap dimensione:", train_bootstrap.shape)


Bootstrap dimensione: (65, 112)


# Feature finali

In [12]:
X_train = prepare_features(train_bootstrap, target_train)
FEATURE_COLUMNS = X_train.columns.tolist()
y_train = train_bootstrap[target_train]

X_val = prepare_features(val_internal, target_train).reindex(columns=FEATURE_COLUMNS)
y_val = val_internal[target_train]

X_test = prepare_features(df_test_external, target_test).reindex(columns=FEATURE_COLUMNS)
y_test = df_test_external.loc[X_test.index, target_test]


# Definisco il modello

In [13]:
model = XGBClassifier(
        random_state=42,
        n_jobs=1,
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0,
        reg_lambda=1,
        scale_pos_weight=1,
    )
model.fit(X_train, y_train);

# validazione DUKE

In [14]:
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

print("\nDUKE – Modello addestarto con training bootstrappato")
print(f"F1 : {f1_score(y_val, y_val_pred):.3f}")
print(f"AUC: {roc_auc_score(y_val, y_val_proba):.3f}")
print(f"ACC: {accuracy_score(y_val, y_val_pred):.3f}")



DUKE – Modello addestarto con training bootstrappato
F1 : 0.833
AUC: 0.936
ACC: 0.882


# Test esterno

In [15]:
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

print("\nAMBL – Validation (bootstrap)")
print(f"F1 : {f1_score(y_test, y_test_pred):.3f}")
print(f"AUC: {roc_auc_score(y_test, y_test_proba):.3f}")
print(f"ACC: {accuracy_score(y_test, y_test_pred):.3f}")



AMBL – Validation (bootstrap)
F1 : 0.631
AUC: 0.492
ACC: 0.460
